In [0]:
FROST_CLIENT_ID = dbutils.secrets.get(scope="weather_price", key="frost_client_id")
print(FROST_CLIENT_ID)

In [0]:
import requests

FROST_CLIENT_ID = dbutils.secrets.get(
    scope="weather_price",
    key="frost_client_id"
)

# 1. Verify electricity price API (NO2)
PRICE_URL = (
    "https://www.hvakosterstrommen.no/"
    "api/v1/prices/2025/01-15_NO2.json"
)
price_response = requests.get(PRICE_URL, timeout=20)
print("Price status:", price_response.status_code)
price_response.raise_for_status()
price_payload = price_response.json()
assert isinstance(price_payload, list) and len(price_payload) > 0
print(price_payload[:2])

# 2. Verify Frost station metadata (SN44560 - Sola)
source_response = requests.get(
    "https://frost.met.no/sources/v0.jsonld",
    params={"ids": "SN44560", "types": "SensorSystem"},
    auth=(FROST_CLIENT_ID, ""),
    timeout=20
)
print("Source status:", source_response.status_code)
source_response.raise_for_status()
print(source_response.json().get("data", []))

# 3. Verify available hourly time series for the target period
timeseries_response = requests.get(
    "https://frost.met.no/observations/availableTimeSeries/v0.jsonld",
    params={
        "sources": "SN44560",
        "elements": (
            "air_temperature,wind_speed,"
            "sum(precipitation_amount PT1H)"
        ),
        "referencetime": "2025-01-01/2025-03-31"
    },
    auth=(FROST_CLIENT_ID, ""),
    timeout=30
)
print("Timeseries status:", timeseries_response.status_code)
timeseries_response.raise_for_status()

available = timeseries_response.json().get("data", [])
for item in available:
    print(
        item.get("sourceId"),
        item.get("elementId"),
        item.get("timeResolution"),
        item.get("timeOffset"),
        item.get("validFrom"),
        item.get("validTo")
    )

required = {"air_temperature", "wind_speed"}
required_hourly = {
    item.get("elementId")
    for item in available
    if item.get("timeResolution") == "PT1H"
}
missing_required = required - required_hourly
if missing_required:
    raise ValueError(
        f"SN44560 is missing required PT1H series: {missing_required}"
    )

print("Preflight passed for NO2 and SN44560.")